# 🇹🇳 Tunisian Arabizi Assistant — Fine-tune + Evaluate (Kaggle)

This notebook **trains** a Tunisian-Derja (Arabizi) assistant on your dataset, **measures** how
good the data is (dialect-rate **before vs after** training), and lets you **chat** with the model.

## Before you run (one-time setup)
1. **Upload your data as a Kaggle Dataset**: the 3 files `cs_pairs.jsonl`, `parallel_pairs.jsonl`,
   `eval_set.jsonl` (drag-and-drop → *+ Add Data* → *New Dataset*). 
2. Right panel → **Accelerator: GPU T4 x2** (or P100). 
3. Right panel → **Internet: ON** (needed to install Unsloth + download the model). 
4. **Run All**. Training a ~10k-pair smoke test takes ~30–60 min on a T4.

> This is a **dataset-efficiency test**. A 7B model is used for a representative result; switch the
> `MODEL_NAME` in the config cell to Qwen3-8B for the real v1, or Qwen2.5-3B for a faster run.


## 1. Install (Unsloth = fast, low-VRAM QLoRA)

In [ ]:
%%capture
# Unsloth handles QLoRA on a single T4 efficiently.
!pip install -q -U unsloth unsloth_zoo
!pip install -q --no-deps trl peft accelerate bitsandbytes
# CRITICAL: transformers v5 breaks Unsloth's training step ("'int' object has no attribute
# 'mean'"). Pin to 4.x. Run this AS THE FIRST CELL, then RESTART the kernel, then Run All.
!pip install -q "transformers<5"
import shutil; shutil.rmtree('/kaggle/working/unsloth_compiled_cache', ignore_errors=True)


## 2. Config — change models / sizes here

In [ ]:
# ---- Model (4-bit, Unsloth). Safe default for free T4. ----
MODEL_NAME       = 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit'
#  alternatives: 'unsloth/Qwen3-8B-bnb-4bit' (real v1, heavier) | 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit' (fast)
MAX_SEQ_LEN      = 1024

# ---- Data mixing (our strategy: conversation is the core; sample the parallel) ----
CONV_UPSAMPLE    = 3       # repeat the hand-written conversational pairs N times (they're the priority)
PARALLEL_SAMPLE  = 8000    # how many translation/comprehension pairs to mix in

# ---- Training ----
EPOCHS           = 2
LR               = 2e-4
BATCH            = 2
GRAD_ACCUM       = 1       # keep at 1: >1 can trigger an Unsloth num_items_in_batch bug

# ---- Eval ----
EVAL_MAX_NEW     = 160

SYSTEM = ('Enti assistant tunsi (service client w 7adith 3am). Jaweb DIMA bel derja tounsiya '
          'bel arabizi (7ourouf latiniya w arqam), b tari9a tabi3iya w 9sira. Ken el user yekteb '
          'bel 3arbi wala faransi wala anglais, efhem w jaweb bel arabizi. Ken talbou translation, '
          'a3mel el li talbou.')
print('config ready ->', MODEL_NAME)


## 3. Load + mix + format the data

In [ ]:
import json, glob, random
random.seed(42)

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits: raise FileNotFoundError(f'{name} not found under /kaggle/input — upload your dataset')
    return hits[0]

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]

conv = load_jsonl(find('cs_pairs.jsonl'))
par  = load_jsonl(find('parallel_pairs.jsonl'))
random.shuffle(par)
par = par[:PARALLEL_SAMPLE]
train_rows = conv * CONV_UPSAMPLE + par
random.shuffle(train_rows)
print(f'conversational: {len(conv)} x{CONV_UPSAMPLE} | parallel sampled: {len(par)} | TOTAL train: {len(train_rows)}')

def to_text(tokenizer, r):
    msgs = [{'role':'system','content':SYSTEM},
            {'role':'user','content':r['instruction']},
            {'role':'assistant','content':r['output']}]
    return tokenizer.apply_chat_template(msgs, tokenize=False)


## 4. Load the model + attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model, r = 16, lora_alpha = 16, lora_dropout = 0,
    target_modules = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing = 'unsloth', random_state = 42,
)
print('model loaded + LoRA attached')


## 5. Helper to chat with the model (used for baseline + final test)

In [ ]:
def generate(user_msg, system=SYSTEM, max_new_tokens=EVAL_MAX_NEW, temperature=0.7):
    FastLanguageModel.for_inference(model)
    msgs = [{'role':'system','content':system},{'role':'user','content':user_msg}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=temperature, top_p=0.9, repetition_penalty=1.1,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


## 6. Dialect-rate metric (MSA-leakage detector, inline)

In [ ]:
import re
TUNISIAN = set('barcha famma chnowa chnoua 9addech 9adech kifech 3lech win wa9tech ya5i 5ouya mte3 '
  'mta3 bch taw tawa ken kima brabi 3aslama 3aslema marhba chwaya barka fissa3 zin behi bahi mli7 '
  '3andi 3andou 3andek hethi hetha hakka haka sa7a 3aychek yezzi 9a3ed mch mouch ma3andich n7eb t7eb '
  '9olli ya3ni fel lel el enti ena houwa hia 9ahwa ghodwa lyoum wala 5dma 5edma dar bnin tounsi'.split())
FRENCH = set('livraison prix commande merci bonjour stock weekend promo garantie couleur taille'.split())
MSA = set('hadha hadhihi alladhi allati sawfa laysa kayfa limadha 3indama ladhalika lakinna jiddan '
  'kathiran yumkinu yajibu na7nu inna sayakun dhalika tilka hunaka faqat aydan ladayna lan lam qad'.split())
MSA_AR = ['الذي','التي','سوف','ليس','كيف','عندما','لذلك','يمكن','يجب','نحن','جدا','هذا','هذه','ذلك']
_AR = re.compile(r'[\u0600-\u06ff]'); _NUM = re.compile(r"[a-z][3-9'][a-z]", re.I); _W = re.compile(r"[a-z0-9'7359]+", re.I)
def score_text(t):
    t=(t or '').strip(); toks=set(w.lower() for w in _W.findall(t))
    tun=len(toks&TUNISIAN)+len(toks&FRENCH)+len(_NUM.findall(t.lower()))
    msa=len(toks&MSA)+sum(t.count(m) for m in MSA_AR)
    if _AR.search(t) and not (toks&TUNISIAN): return 'arabic_script'
    if tun==0 and msa==0: return 'unknown'
    if msa>0 and tun==0: return 'msa_leak'
    if tun>0 and msa==0: return 'tunisian'
    return 'tunisian' if tun>=2*msa else 'mixed'
def dialect_report(preds, tag=''):
    from collections import Counter
    c=Counter(score_text(p) for p in preds); n=len(preds) or 1
    print(f'--- {tag} (n={len(preds)}) ---')
    for k in ['tunisian','mixed','msa_leak','arabic_script','unknown']:
        print(f'  {k:14}: {c.get(k,0):4}  ({c.get(k,0)/n:.0%})')
    rate=c.get('tunisian',0)/n; print(f'  >> DIALECT RATE: {rate:.0%}')
    return rate


## 7. BASELINE — how Tunisian is the model *before* training?
(Run the untrained model on the eval prompts. Expect a LOW dialect rate — it'll lean MSA/English.)

In [ ]:
eval_rows = load_jsonl(find('eval_set.jsonl'))
print(f'eval items: {len(eval_rows)}')
base_preds = [generate(r['instruction']) for r in eval_rows]
base_rate = dialect_report(base_preds, 'BASELINE (before fine-tuning)')


## 8. Train (QLoRA / SFT)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset

FastLanguageModel.for_training(model)
ds = Dataset.from_dict({'text': [to_text(tokenizer, r) for r in train_rows]})

trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    dataset_text_field = 'text', max_seq_length = MAX_SEQ_LEN, packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = BATCH, gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps = 10, num_train_epochs = EPOCHS, learning_rate = LR,
        fp16 = not torch.cuda.is_bf16_supported(), bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 20, optim = 'adamw_8bit', weight_decay = 0.01,
        lr_scheduler_type = 'linear', seed = 42, output_dir = 'outputs', report_to = 'none',
    ),
)
# NOTE: 'train_on_responses_only' can trigger \"'int' object has no attribute 'mean'\" on some
# transformers/trl versions (it masks the batch oddly). Default OFF -> full-text SFT, robust.
USE_RESPONSES_ONLY = False
if USE_RESPONSES_ONLY:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(trainer,
        instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')

trainer.train()


## 9. Save the LoRA adapter (download it after the run)

In [ ]:
model.save_pretrained('/kaggle/working/tunisian_lora')
tokenizer.save_pretrained('/kaggle/working/tunisian_lora')
print('saved -> /kaggle/working/tunisian_lora  (download from the Output tab)')


## 10. AFTER — measure dialect rate *after* training (the dataset-efficiency result)

In [ ]:
after_preds = [generate(r['instruction']) for r in eval_rows]
after_rate = dialect_report(after_preds, 'AFTER fine-tuning')
print(f'\n=== DATASET EFFICIENCY ===')
print(f'dialect rate  BEFORE: {base_rate:.0%}   AFTER: {after_rate:.0%}   (+{(after_rate-base_rate)*100:.0f} pts)')
print('\n--- 8 sample before/after ---')
for r, b, a in list(zip(eval_rows, base_preds, after_preds))[:8]:
    print('Q  :', r['instruction'][:70])
    print('OLD:', b[:90]); print('NEW:', a[:90]); print()


## 11. Talk to your model 🇹🇳
Edit the message and re-run. It understands Arabizi, Arabic letters, French, and English.

In [ ]:
for msg in ['3aslema chna7welek?',
            'قداش تمن التوصيل لصفاقس؟',
            'give me a healthy breakfast idea',
            '9olli nokta tdha7ek',
            'a7kili 7keya 9sira 3la el sabr']:
    print('🧑', msg)
    print('🤖', generate(msg), '\n')


In [ ]:
# >>> your turn: change this and re-run <<<
print(generate('3andi mochkla fel commande mte3i, chnowa na3mel?'))


## 12. What next
- **Read the BEFORE→AFTER dialect rate** above — that's your dataset efficiency. A big jump = the data works.
- Look at the sample answers: are they natural Tunisian? note weak categories.
- **Download** `/kaggle/working/tunisian_lora` (Output tab) — that's your trained adapter.
- Improve the weakest axis (more pairs there), regenerate, re-run. That's the measure→improve loop.
- For the real v1: set `MODEL_NAME='unsloth/Qwen3-8B-bnb-4bit'` and grow the data.
